# RetainIQ — Phase 3.2: MySQL Database Build & Data Loading

## Objective

I will build the RetainIQ database in MySQL 8.x and load the cleaned dataset through a staging table. I use `mysql-connector-python` so the Jupyter notebook communicates directly with MySQL.

## 1. Load the Dataset and Connect to MySQL

In [3]:
from pathlib import Path
from getpass import getpass
import pandas as pd
import mysql.connector
from mysql.connector import Error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

MYSQL_CONFIG = {"host":"localhost","port":3306,"user":"retainiq_user","password":getpass("Enter MySQL password for retainiq_user: "),"database":"retainiq"}

def get_connection(): return mysql.connector.connect(**MYSQL_CONFIG)

def run_query(query, params=None):
    connection=cursor=None
    try:
        connection=get_connection(); cursor=connection.cursor(dictionary=True); cursor.execute(query, params or ()); return pd.DataFrame(cursor.fetchall())
    except Error as exc:
        print(f"MySQL error: {exc}"); return None
    finally:
        if cursor: cursor.close()
        if connection and connection.is_connected(): connection.close()

print("MySQL helper functions are ready.")

Loaded cleaned dataset: 7,043 rows × 51 columns
MySQL helper functions are ready.


## 2. Test the MySQL Connection

In [4]:
connection=None
try:
    connection=get_connection(); cursor=connection.cursor(); cursor.execute("SELECT VERSION()"); print(f"Connected successfully. MySQL version: {cursor.fetchone()[0]}")
except Error as exc:
    raise RuntimeError(f"MySQL connection failed: {exc}") from exc
finally:
    if connection and connection.is_connected():
        cursor.close(); connection.close()

Connected successfully. MySQL version: 8.0.42


## 3. Create the Database

In [5]:
server=mysql.connector.connect(host=MYSQL_CONFIG["host"],port=MYSQL_CONFIG["port"],user=MYSQL_CONFIG["user"],password=MYSQL_CONFIG["password"])
cursor=server.cursor(); cursor.execute("CREATE DATABASE IF NOT EXISTS retainiq"); server.commit(); cursor.close(); server.close(); print("Database `retainiq` is ready.")

Database `retainiq` is ready.


## 4. Create the Staging Table

In [6]:
staging_sql = """
CREATE TABLE IF NOT EXISTS stg_telco_clean (
    `Customer ID` VARCHAR(20), `Gender` VARCHAR(20), `Age` SMALLINT,
    `Under 30` VARCHAR(10), `Senior Citizen` VARCHAR(10), `Married` VARCHAR(10),
    `Dependents` VARCHAR(10), `Number of Dependents` SMALLINT,
    `Country` VARCHAR(100), `State` VARCHAR(100), `City` VARCHAR(150), `Zip Code` INT,
    `Latitude` DECIMAL(9,6), `Longitude` DECIMAL(9,6), `Population` INT,
    `Quarter` VARCHAR(20), `Referred a Friend` VARCHAR(10), `Number of Referrals` INT,
    `Tenure in Months` SMALLINT, `Offer` VARCHAR(100), `Phone Service` VARCHAR(20),
    `Avg Monthly Long Distance Charges` DECIMAL(12,2), `Multiple Lines` VARCHAR(30),
    `Internet Service` VARCHAR(20), `Internet Type` VARCHAR(40), `Avg Monthly GB Download` INT,
    `Online Security` VARCHAR(20), `Online Backup` VARCHAR(20), `Device Protection Plan` VARCHAR(30),
    `Premium Tech Support` VARCHAR(30), `Streaming TV` VARCHAR(20), `Streaming Movies` VARCHAR(20),
    `Streaming Music` VARCHAR(20), `Unlimited Data` VARCHAR(20), `Contract` VARCHAR(30),
    `Paperless Billing` VARCHAR(20), `Payment Method` VARCHAR(50), `Monthly Charge` DECIMAL(12,2),
    `Total Charges` DECIMAL(14,2), `Total Refunds` DECIMAL(14,2), `Total Extra Data Charges` DECIMAL(14,2),
    `Total Long Distance Charges` DECIMAL(14,2), `Total Revenue` DECIMAL(14,2), `Satisfaction Score` TINYINT,
    `Customer Status` VARCHAR(30), `Churn Label` VARCHAR(10), `Churn Score` TINYINT, `CLTV` INT,
    `Churn Category` VARCHAR(100), `Churn Reason` VARCHAR(200), `is_new_customer` BOOLEAN
) ENGINE=InnoDB;
"""
connection=get_connection(); cursor=connection.cursor(); cursor.execute(staging_sql); connection.commit(); cursor.close(); connection.close(); print("Staging table created.")

Staging table created.


## 5. Load the Cleaned CSV Into MySQL

I use a parameterized `executemany()` load through `mysql-connector-python`. This avoids the `pandas.read_sql` warning and keeps MySQL as the database performing the inserts. With 7,043 rows, this approach is fast enough for this project.

In [7]:
staging_columns=["Customer ID","Gender","Age","Under 30","Senior Citizen","Married","Dependents","Number of Dependents","Country","State","City","Zip Code","Latitude","Longitude","Population","Quarter","Referred a Friend","Number of Referrals","Tenure in Months","Offer","Phone Service","Avg Monthly Long Distance Charges","Multiple Lines","Internet Service","Internet Type","Avg Monthly GB Download","Online Security","Online Backup","Device Protection Plan","Premium Tech Support","Streaming TV","Streaming Movies","Streaming Music","Unlimited Data","Contract","Paperless Billing","Payment Method","Monthly Charge","Total Charges","Total Refunds","Total Extra Data Charges","Total Long Distance Charges","Total Revenue","Satisfaction Score","Customer Status","Churn Label","Churn Score","CLTV","Churn Category","Churn Reason","is_new_customer"]
placeholders=", ".join(["%s"]*len(staging_columns))
columns=", ".join(f"`{c}`" for c in staging_columns)
insert_sql=f"INSERT INTO stg_telco_clean ({columns}) VALUES ({placeholders})"
records=[tuple(None if pd.isna(v) else v for v in row) for row in df[staging_columns].itertuples(index=False,name=None)]
connection=get_connection(); cursor=connection.cursor(); cursor.execute("TRUNCATE TABLE stg_telco_clean"); cursor.executemany(insert_sql,records); connection.commit(); print(f"Loaded {cursor.rowcount:,} staging rows."); cursor.close(); connection.close()

Loaded 7,043 staging rows.


## 6. Validate the Staging Load

In [8]:
staging_check=run_query("""SELECT COUNT(*) AS rows_loaded, COUNT(DISTINCT `Customer ID`) AS unique_customers FROM stg_telco_clean;""")
staging_check

,rows_loaded,unique_customers
0,7043,7043


In [9]:
assert int(staging_check.loc[0,"rows_loaded"])==7043
assert int(staging_check.loc[0,"unique_customers"])==7043
print("PASS — All cleaned customer records loaded into staging.")

PASS — All cleaned customer records loaded into staging.


## 7. Create the Fact Table

In [10]:
fact_sql="""CREATE TABLE IF NOT EXISTS fact_customer_status (customer_id VARCHAR(20) NOT NULL, tenure_months SMALLINT NOT NULL, monthly_charge DECIMAL(12,2) NOT NULL, total_charges DECIMAL(14,2) NOT NULL, total_refunds DECIMAL(14,2) NOT NULL, total_revenue DECIMAL(14,2) NOT NULL, satisfaction_score TINYINT NOT NULL, churn_score TINYINT NOT NULL, cltv INT NOT NULL, churn_label VARCHAR(10) NOT NULL, customer_status VARCHAR(30) NOT NULL, PRIMARY KEY(customer_id)) ENGINE=InnoDB;"""
connection=get_connection(); cursor=connection.cursor(); cursor.execute(fact_sql); connection.commit(); cursor.close(); connection.close(); print("Fact table created.")

Fact table created.


## 8. Create the Five Dimension Tables

In [11]:
dimensions_sql="""
CREATE TABLE IF NOT EXISTS dim_demographics (customer_id VARCHAR(20) NOT NULL, gender VARCHAR(20), age SMALLINT, under_30 VARCHAR(10), senior_citizen VARCHAR(10), married VARCHAR(10), dependents VARCHAR(10), number_of_dependents SMALLINT, PRIMARY KEY(customer_id), FOREIGN KEY(customer_id) REFERENCES fact_customer_status(customer_id)) ENGINE=InnoDB;
CREATE TABLE IF NOT EXISTS dim_location (customer_id VARCHAR(20) NOT NULL, country VARCHAR(100), state VARCHAR(100), city VARCHAR(150), zip_code INT, latitude DECIMAL(9,6), longitude DECIMAL(9,6), population INT, PRIMARY KEY(customer_id), FOREIGN KEY(customer_id) REFERENCES fact_customer_status(customer_id)) ENGINE=InnoDB;
CREATE TABLE IF NOT EXISTS dim_services (customer_id VARCHAR(20) NOT NULL, phone_service VARCHAR(20), multiple_lines VARCHAR(30), internet_service VARCHAR(20), internet_type VARCHAR(40), online_security VARCHAR(20), online_backup VARCHAR(20), device_protection_plan VARCHAR(30), premium_tech_support VARCHAR(30), streaming_tv VARCHAR(20), streaming_movies VARCHAR(20), streaming_music VARCHAR(20), unlimited_data VARCHAR(20), avg_monthly_long_distance_charges DECIMAL(12,2), avg_monthly_gb_download INT, PRIMARY KEY(customer_id), FOREIGN KEY(customer_id) REFERENCES fact_customer_status(customer_id)) ENGINE=InnoDB;
CREATE TABLE IF NOT EXISTS dim_account (customer_id VARCHAR(20) NOT NULL, quarter VARCHAR(20), referred_a_friend VARCHAR(10), number_of_referrals INT, offer VARCHAR(100), contract VARCHAR(30), paperless_billing VARCHAR(20), payment_method VARCHAR(50), total_extra_data_charges DECIMAL(14,2), total_long_distance_charges DECIMAL(14,2), PRIMARY KEY(customer_id), FOREIGN KEY(customer_id) REFERENCES fact_customer_status(customer_id)) ENGINE=InnoDB;
CREATE TABLE IF NOT EXISTS dim_churn_detail (customer_id VARCHAR(20) NOT NULL, churn_category VARCHAR(100), churn_reason VARCHAR(200), PRIMARY KEY(customer_id), FOREIGN KEY(customer_id) REFERENCES fact_customer_status(customer_id)) ENGINE=InnoDB;
"""
connection=get_connection(); cursor=connection.cursor(); [cursor.execute(s.strip()) for s in dimensions_sql.split(';') if s.strip()]; connection.commit(); cursor.close(); connection.close(); print("All five dimension tables created.")

All five dimension tables created.


## 9. Populate Fact and Dimensions

In [13]:
# Clear existing data safely.
# I disable foreign-key checks temporarily because the tables reference
# fact_customer_status through foreign keys.

connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")

    for table in [
        "dim_demographics",
        "dim_location",
        "dim_services",
        "dim_account",
        "dim_churn_detail",
        "fact_customer_status"
    ]:
        cursor.execute(f"TRUNCATE TABLE {table}")

    cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")

    # Populate fact table first.
    cursor.execute("""
        INSERT INTO fact_customer_status (
            customer_id,
            tenure_months,
            monthly_charge,
            total_charges,
            total_refunds,
            total_revenue,
            satisfaction_score,
            churn_score,
            cltv,
            churn_label,
            customer_status
        )
        SELECT
            `Customer ID`,
            `Tenure in Months`,
            `Monthly Charge`,
            `Total Charges`,
            `Total Refunds`,
            `Total Revenue`,
            `Satisfaction Score`,
            `Churn Score`,
            `CLTV`,
            `Churn Label`,
            `Customer Status`
        FROM stg_telco_clean;
    """)

    # Populate dimensions after the fact table.
    cursor.execute("""
        INSERT INTO dim_demographics
        SELECT
            `Customer ID`,
            `Gender`,
            `Age`,
            `Under 30`,
            `Senior Citizen`,
            `Married`,
            `Dependents`,
            `Number of Dependents`
        FROM stg_telco_clean;
    """)

    cursor.execute("""
        INSERT INTO dim_location
        SELECT
            `Customer ID`,
            `Country`,
            `State`,
            `City`,
            `Zip Code`,
            `Latitude`,
            `Longitude`,
            `Population`
        FROM stg_telco_clean;
    """)

    cursor.execute("""
        INSERT INTO dim_services
        SELECT
            `Customer ID`,
            `Phone Service`,
            `Multiple Lines`,
            `Internet Service`,
            `Internet Type`,
            `Online Security`,
            `Online Backup`,
            `Device Protection Plan`,
            `Premium Tech Support`,
            `Streaming TV`,
            `Streaming Movies`,
            `Streaming Music`,
            `Unlimited Data`,
            `Avg Monthly Long Distance Charges`,
            `Avg Monthly GB Download`
        FROM stg_telco_clean;
    """)

    cursor.execute("""
        INSERT INTO dim_account
        SELECT
            `Customer ID`,
            `Quarter`,
            `Referred a Friend`,
            `Number of Referrals`,
            `Offer`,
            `Contract`,
            `Paperless Billing`,
            `Payment Method`,
            `Total Extra Data Charges`,
            `Total Long Distance Charges`
        FROM stg_telco_clean;
    """)

    cursor.execute("""
        INSERT INTO dim_churn_detail
        SELECT
            `Customer ID`,
            `Churn Category`,
            `Churn Reason`
        FROM stg_telco_clean;
    """)

    connection.commit()

    print("Fact and dimension tables populated successfully.")

except Exception:
    connection.rollback()
    raise

finally:
    cursor.close()
    connection.close()

Fact and dimension tables populated successfully.


In [14]:
tables = run_query("""
SELECT
    'stg_telco_clean' AS table_name,
    COUNT(*) AS row_count
FROM stg_telco_clean

UNION ALL

SELECT 'fact_customer_status', COUNT(*)
FROM fact_customer_status

UNION ALL

SELECT 'dim_demographics', COUNT(*)
FROM dim_demographics

UNION ALL

SELECT 'dim_location', COUNT(*)
FROM dim_location

UNION ALL

SELECT 'dim_services', COUNT(*)
FROM dim_services

UNION ALL

SELECT 'dim_account', COUNT(*)
FROM dim_account

UNION ALL

SELECT 'dim_churn_detail', COUNT(*)
FROM dim_churn_detail;
""")

tables

,table_name,row_count
0,stg_telco_clean,7043
1,fact_customer_status,7043
2,dim_demographics,7043
3,dim_location,7043
4,dim_services,7043
5,dim_account,7043
6,dim_churn_detail,7043


# Phase 3.2 Conclusion

I have built the MySQL database, loaded the cleaned dataset into staging, created the fact and dimensions, and populated the analytical model directly through MySQL.

**Next:** validate the database and perform analytical SQL.